In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_future_values(principal:float = 10000,
                   simple_interest: float = 0.1,
                   compound_interest: float = 0.1,
                   periods: int = 20):

  assert 0 < simple_interest < 1, f"Interest rate should be between 0 and 1, received {simple_interest}"
  assert 0 < compound_interest < 1, f"Interest rate should be between 0 and 1, received {compound_interest}"
  assert type(periods) is int, f"Periods should be an integer"
  assert periods > 0, f"Periods should be a positive integer"

  # 2. Generate Data
  n = np.arange(0, periods + 1, 1)
  F_simple = principal * (1 + simple_interest * n)
  F_compound = principal * (1 + compound_interest)**n

  I_simple = principal * simple_interest * np.ones(periods + 1)
  I_simple[0] = 0
  
  # Calculate Compound Interest Breakdown
  I_compound_principal = principal * compound_interest * np.ones(periods + 1)
  I_compound_principal[0] = 0
  I_compound_on_interest = np.zeros(periods + 1)
  
  for k in range(1, periods + 1):
    I_compound_on_interest[k] = (F_compound[k-1] - principal) * compound_interest


  # 3. Build the Interactive Plot
  fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=('Future Value', 'Interest Earned Per Period')
    )
  # --- ROW 1: Cumulative Lines ---
  # Add Simple Interest Line
  fig.add_trace(go.Scatter(
      x=n, y=F_simple,
      mode='lines+markers',
      name='Total: Simple',
      line=dict(color='blue', dash='dash'),
      hovertemplate='Year %{x}<br>Amount: $%{y:,.2f}<extra></extra>'
  ), row=1, col=1)

  # Add Compound Interest Line
  fig.add_trace(go.Scatter(
      x=n, y=F_compound,
      mode='lines+markers',
      name='Total: Compound',
      line=dict(color='red'),
      hovertemplate='Year %{x}<br>Amount: $%{y:,.2f}<extra></extra>'
  ), row=1, col=1)

  # --- ROW 2: Period Interest Bars ---
  # Add Simple Interest Bars
  fig.add_trace(go.Bar(
      x=n, y=I_simple,
      name='Period: Simple',
      marker_color='blue',
      opacity=0.6,
      offsetgroup=0,
      hovertemplate='Year %{x} Interest: $%{y:,.2f}<extra></extra>'
  ), row=2, col=1)

  # Add Compound Interest Bars (On Principal)
  fig.add_trace(go.Bar(
      x=n, y=I_compound_principal,
      name='Compound: On Principal',
      marker_color='red',
      opacity=0.6,
      offsetgroup=1,
      hovertemplate='Year %{x} On Principal: $%{y:,.2f}<extra></extra>'
  ), row=2, col=1)

  # Add Compound Interest Bars (On Accumulated Interest)
  fig.add_trace(go.Bar(
      x=n, y=I_compound_on_interest,
      name='Compound: On Interest',
      marker_color='darkred',
      opacity=0.8,
      offsetgroup=1,
      base=I_compound_principal,
      hovertemplate='Year %{x} On Interest: $%{y:,.2f}<extra></extra>'
  ), row=2, col=1)

  # 4. Formatting the layout
  fig.update_layout(
      title='Simple vs. Compound Interest Analysis',
      hovermode='x unified', # Shows all values simultaneously on hover
      template='plotly_white',
      barmode='group',       # Groups the bars side-by-side
      height=700             # Increased height to accommodate both plots
  )

  # Update axis labels
  fig.update_yaxes(title_text="Total Value ($)", row=1, col=1)
  fig.update_yaxes(title_text="Interest Value ($)", row=2, col=1)
  fig.update_xaxes(title_text="Number of Periods (periods)", row=2, col=1)

  fig.show()